In [0]:
# ─── CÉLULA 1 — CONFIGURAÇÃO E IMPORTAÇÕES ───────────────────────────────────────
# Define constantes e importa as classes do Spark ML necessárias para o pipeline.
# A arquitetura de Pipeline do Spark ML encadeia transformações de forma reproduzível: fit() aprende os parâmetros (vocabulário de strings, estatísticas), transform() aplica as transformações — o mesmo objeto pode ser salvo e reutilizado no notebook 06.

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.sql import functions as F

GOLD_TABLE = "workspace.default.telco_gold"

# Colunas categóricas que receberão StringIndexer + OneHotEncoder
CAT_COLS = [
    "gender", "SeniorCitizen", "Partner", "Dependents",
    "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod"
]

# Colunas numéricas originais — entram diretamente no VectorAssembler
NUM_COLS = ["tenure", "MonthlyCharges", "TotalCharges"]

df = spark.read.table("workspace.default.telco_silver")
print(f"✔ Prata carregada: {df.count()} linhas")

In [0]:
# ─── CÉLULA 2 — CRIAÇÃO DE FEATURES DERIVADAS ────────────────────────────────────
# Três features de domínio criadas manualmente — não geradas pelo pipeline Spark ML.
# IMPORTANTE: estas features devem ser recriadas antes de qualquer chamada a pipeline_loaded.transform() em dados novos (ver notebook 06, célula 4).

# num_services: contagem de serviços adicionais (0–8). O EDA (notebook 03) mostrou correlação negativa com churn — mais serviços, maior retenção.

# charges_per_tenure: TotalCharges / tenure = custo médio mensal acumulado. Para tenure=0, usa MonthlyCharges para evitar divisão por zero. Captura o valor percebido pelo cliente ao longo do tempo.

# is_new_customer: flag binária para os primeiros 12 meses (período crítico identificado no EDA com ~47.7% de taxa de churn).

service_cols = [
    "PhoneService", "MultipleLines", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies"
]

df = df.withColumn(
    "num_services",
    sum([F.when(F.col(c) == "Yes", 1).otherwise(0) for c in service_cols])
)

df = df.withColumn(
    "charges_per_tenure",
    F.when(
        F.col("tenure") > 0,
        F.col("TotalCharges") / F.col("tenure")
    ).otherwise(F.col("MonthlyCharges"))
)

df = df.withColumn(
    "is_new_customer",
    F.when(F.col("tenure") <= 12, 1).otherwise(0)
)

display(df.select("tenure", "num_services", "charges_per_tenure", "is_new_customer").limit(5))

In [0]:
# ─── CÉLULA 3 — STRINGINDEXER: CATEGÓRICAS → ÍNDICES NUMÉRICOS ───────────────────
# StringIndexer converte cada coluna categórica em índices inteiros ordenados por frequência (o valor mais comum recebe índice 0).
# handleInvalid="keep" garante que valores desconhecidos em produção (ex: nova categoria não vista no treino) recebam um índice extra em vez de causar erro.
# As novas features numéricas binárias são adicionadas a NUM_COLS aqui.

NUM_COLS += ["num_services", "charges_per_tenure", "is_new_customer"]

indexers = [
    StringIndexer()
    .setInputCol(c)
    .setOutputCol(f"{c}_idx")
    .setHandleInvalid("keep")
    for c in CAT_COLS
]

print(f"✔ {len(indexers)} StringIndexers criados com sucesso (padrão Setters)")

print(f"✔ {len(indexers)} StringIndexers criados")
print("  Exemplo de saída: gender → gender_idx, Contract → Contract_idx ...")

---------------------------------------------------------------------------
Py4JError                                 Traceback (most recent call last)
File <command-6420681934094607>, line 8
      5 from pyspark.ml.feature import StringIndexer
      6 NUM_COLS += ["num_services", "charges_per_tenure", "is_new_customer"]
----> 8 indexers = [
      9     StringIndexer()
     10     .setInputCol(c)
     11     .setOutputCol(f"{c}_idx")
     12     .setHandleInvalid("keep")
     13     for c in CAT_COLS
     14 ]
     16 print(f"✔ {len(indexers)} StringIndexers criados com sucesso (padrão Setters)")
     18 print(f"✔ {len(indexers)} StringIndexers criados")

File <command-6420681934094607>, line 9, in <listcomp>(.0)
      5 from pyspark.ml.feature import StringIndexer
      6 NUM_COLS += ["num_services", "charges_per_tenure", "is_new_customer"]
      8 indexers = [
----> 9     StringIndexer()
     10     .setInputCol(c)
     11     .setOutputCol(f"{c}_idx")
     12     .setHandleInvalid("

In [0]:
# ─── CÉLULA 4 — ONEHOTENCODER: ÍNDICES → VETORES BINÁRIOS ESPARSOS ───────────────
# OneHotEncoder elimina a ordem artificial introduzida pelo StringIndexer.
# Sem este passo, o modelo interpretaria Contract_idx: 0 < 1 < 2 como grandeza numérica, o que não faz sentido para variáveis nominais.
# dropLast=True remove uma categoria por variável para evitar multicolinearidade (dummy variable trap) — a categoria descartada é inferida pelo modelo.

encoders = [
    OneHotEncoder(
        inputCol=f"{c}_idx",
        outputCol=f"{c}_ohe",
        dropLast=True
    )
    for c in CAT_COLS
]

print(f"✔ {len(encoders)} OneHotEncoders criados")

---------------------------------------------------------------------------
Py4JError                                 Traceback (most recent call last)
File <command-6420681934094608>, line 8
      1 # ─── CÉLULA 4 — ONEHOTENCODER: ÍNDICES → VETORES BINÁRIOS ESPARSOS ───────────────
      2 # OneHotEncoder elimina a ordem artificial introduzida pelo StringIndexer.
      3 # Sem este passo, o modelo interpretaria Contract_idx: 0 < 1 < 2 como
      4 # grandeza numérica, o que não faz sentido para variáveis nominais.
      5 # dropLast=True remove uma categoria por variável para evitar multicolinearidade
      6 # (dummy variable trap) — a categoria descartada é inferida pelo modelo.
----> 8 encoders = [
      9     OneHotEncoder(
     10         inputCol=f"{c}_idx",
     11         outputCol=f"{c}_ohe",
     12         dropLast=True
     13     )
     14     for c in CAT_COLS
     15 ]
     17 print(f"✔ {len(encoders)} OneHotEncoders criados")

File <command-6420681934094608>, line 9, i

In [0]:
# ─── CÉLULA 5 — VECTORASSEMBLER: UNIFICAÇÃO DAS FEATURES ─────────────────────────
# VectorAssembler concatena todas as colunas de entrada em um único vetor denso, formato obrigatório para qualquer algoritmo do Spark ML (featuresCol).
# A ordem é: features numéricas originais + novas features + colunas OHE.
# handleInvalid="keep" substitui NaN/null por zero no vetor final em vez de falhar.

ohe_cols       = [f"{c}_ohe" for c in CAT_COLS]
assembler_cols = NUM_COLS + ohe_cols

assembler = VectorAssembler(
    inputCols=assembler_cols,
    outputCol="features",
    handleInvalid="keep"
)

print(f"✔ VectorAssembler criado com {len(assembler_cols)} colunas de entrada")

In [0]:
# ─── CÉLULA 6 — AJUSTE DO PIPELINE E GERAÇÃO DA CAMADA OURO ──────────────────────
# Pipeline.fit() percorre todos os stages em ordem: cada StringIndexer aprende
# o vocabulário da sua coluna, cada OHE aprende as dimensões, o Assembler é stateless.
# Pipeline.transform() aplica todas as transformações em sequência no DataFrame completo.
# O DataFrame final retém apenas `features` (vetor ML) e `label` (Churn binarizado),
# que são as únicas colunas consumidas pelos algoritmos de classificação.

pipeline = Pipeline(stages=indexers + encoders + [assembler])
pipeline_model = pipeline.fit(df)
df_gold = pipeline_model.transform(df)
df_gold = df_gold.select("features", F.col("Churn").alias("label"))

print(f"✔ Pipeline executado — {df_gold.count()} linhas transformadas")
display(df_gold.limit(3))

In [0]:
# ─── CÉLULA 7 — PERSISTÊNCIA DA CAMADA OURO E DO PIPELINE ────────────────────────
# Salva o DataFrame Gold no Unity Catalog e o PipelineModel em um Volume.
# O Volume (/Volumes/) é o caminho correto no Databricks Serverless — o DBFS root (/dbfs/ ou /delta/) está desabilitado neste ambiente.
# Salvar o pipeline permite reutilizá-lo no notebook 06 para transformar dados novos sem precisar re-executar o fit(), garantindo consistência total entre o pré-processamento do treino e o da inferência em produção.

PIPELINE_PATH = "/Volumes/workspace/default/modelos_ml/pipeline_model"

(
    df_gold
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)

pipeline_model.save(PIPELINE_PATH)

print(f"✔ Tabela salva no UC: {GOLD_TABLE}")
print(f"✔ Pipeline salvo em: {PIPELINE_PATH}")

In [0]:
# ─── CÉLULA 8 — VALIDAÇÃO DA CAMADA OURO ─────────────────────────────────────────
# Valida o resultado antes de avançar para o treinamento:
# 1. Contagem de linhas: deve ser 7.043 (sem perda de registros).
# 2. Colunas: apenas ['features', 'label'] — nenhuma feature leaked ao modelo.
# 3. Dimensão do vetor: ~35 features após encoding (16 CAT × OHE + 6 NUM).
# 4. Distribuição do label: confirma desbalanceamento (~26% churn) para embasar a escolha de AUC-ROC como métrica principal no notebook 05.

df_check = spark.read.table(GOLD_TABLE)

print(f"✔ Linhas na Ouro       : {df_check.count()}  (esperado: 7043)")
print(f"✔ Colunas              : {df_check.columns}  (esperado: ['features', 'label'])")

primeira_linha = df_check.first()
print(f"✔ Dimensões do vetor   : {len(primeira_linha['features'])}  (esperado: ~35)")

display(df_check.groupBy("label").count().orderBy("label"))